In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np

# --- 粘贴昨天的核心类 ---
class StockDataset(Dataset):
    def __init__(self, df, window_size=30):
        self.df = df.sort_values('date').reset_index(drop=True)#日期排序
        self.window_size = window_size
        self.features = self.df[['close', 'volume']].values#只留收盘价和成交量
        self.target = self.df['close'].pct_change().shift(-1).values#pct_change()计算收益率，shift(-1)预测未来

    def __len__(self):
        return len(self.df) - self.window_size - 1#总天数 - 30天(观察期) - 1天(未来的答案)

    def __getitem__(self, idx):
        x = self.features[idx : idx + self.window_size]
        y = self.target[idx + self.window_size]
        # 记得把 y 包在列表里，方便模型识别
        return torch.FloatTensor(x), torch.FloatTensor([y])

In [ ]:
!pip install --upgrade bottleneck --user

In [2]:
# 1. 造 100 天的假数据
dummy_data = {
    'date': pd.date_range(start='2020-01-01', periods=100),
    'close': np.random.rand(100) * 100,
    'volume': np.random.randint(1000, 5000, 100)
}
df = pd.DataFrame(dummy_data)

# 2. 实例化数据集 (窗口选30天)
train_ds = StockDataset(df, window_size=30)

# 3. 召唤DataLoader，每次8 个案例
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)

print(f"一共有 {len(train_ds)} 个案例，每批次 8 个。")

一共有 69 个案例，每批次 8 个。


In [3]:
import torch.nn as nn

class SimpleQuantModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        # 采购两台机器：
        # 60个输入，16个隐藏特征
        self.fc1 = nn.Linear(input_dim, 16)
        # 激活函数：给模型增加一点“直觉”，让它能理解非线性波动
        self.relu = nn.ReLU()
        # 把 16 个特征压缩成 1 个预测值（明天的收益率）
        self.fc2 = nn.Linear(16, 1)
        
    def forward(self, x):
        # 1. 把搬运工送来的 [8, 30, 2] 拍平成 [8, 60]
        x = x.view(x.shape[0], -1)
        # 2. 依次经过
        x = self.fc1(x)
        x = self.relu(x)
        out = self.fc2(x)
        return out

# 实例化模型：输入维度是 30天 * 2个特征 = 60
model = SimpleQuantModel(input_dim=60)
print("--- 第一个量化模型结构 ---")
print(model)

--- 第一个量化模型结构 ---
SimpleQuantModel(
  (fc1): Linear(in_features=60, out_features=16, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=16, out_features=1, bias=True)
)


In [4]:
# 1. 模拟从搬运工那里拿一车货
# 这里的 train_loader 是你刚才定义的 DataLoader
batch_x, batch_y = next(iter(train_loader))

# 2. 把这车货塞进我们的“加工工厂”
# 模型会自动调用 forward 方法
prediction = model(batch_x)

print("--- 压力测试成功！ ---")
print(f"输入形状: {batch_x.shape} (8个案例, 每个看30天, 2个特征)")
print(f"输出预测形状: {prediction.shape} (得到8个预测结果)")
print(f"具体的预测数值（前3个）:\n{prediction[:3]}")

--- 压力测试成功！ ---
输入形状: torch.Size([8, 30, 2]) (8个案例, 每个看30天, 2个特征)
输出预测形状: torch.Size([8, 1]) (得到8个预测结果)
具体的预测数值（前3个）:
tensor([[ 121.8031],
        [ 131.5992],
        [-217.3645]], grad_fn=<SliceBackward0>)


In [5]:
import torch
import numpy as np
import random

# 1. 第一步：先念定身咒（设置所有种子）
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# 2. 第二步：造一份“固定的”假数据
data = np.random.randn(100, 2) # 100天，2个特征
df = pd.DataFrame(data, columns=['close', 'volume'])
df['date'] = pd.date_range('2020-01-01', periods=100)

# 3. 第三步：买机器（实例化模型）
# 此时因为种子定了，fc1 和 fc2 里面的随机权重就被死死锁在 42 号页面了
test_model = SimpleQuantModel(input_dim=60)

# 4. 第四步：拿货（不许乱序）
test_ds = StockDataset(df, window_size=30)
# 注意这里 shuffle=False，保证每次拿的都是前 8 个
test_loader = DataLoader(test_ds, batch_size=8, shuffle=False)

# 5. 第五步：看结果
batch_x, _ = next(iter(test_loader))
pred = test_model(batch_x)
print(pred[:3]) # 连跑两次这个 Cell，看这三个数动不动

tensor([[-0.1368],
        [ 0.2313],
        [-0.0306]], grad_fn=<SliceBackward0>)
